## Gather links to publications since 2015

In [168]:
import re
from urllib.request import urlopen
from urllib.request import Request
from urllib.error import HTTPError, URLError
from urllib.parse import urljoin
from bs4 import BeautifulSoup

base_url = "https://scholar.google.com/citations?user=ETIBghkAAAAJ&hl=en"
url_limit = 99

class webscraper:
    def get_bs_from_url(url = base_url):
        try:
            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
                "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,image/apng,*/*;q=0.8",
                "Accept-Encoding": "gzip, deflate, br",
                "Accept-Language": "en-US,en;q=0.9",
                "Connection": "keep-alive",
                "Upgrade-Insecure-Requests": "1"
            }
           
            response = urlopen(Request(url, headers=headers))
            # To avoid 'Too many requests'
            time.sleep(5)
            return BeautifulSoup(response.read(), 'html.parser')
        
        except HTTPError as e:
            global url_limit
            url_limit = 3
            print(f"HTTP error: {e.code} - {e.reason}")
            print("Loading from file")

            if url == base_url:
                filename = "home"
            else:
                match = re.search(r"citation_for_view=[^:]+:([a-zA-Z0-9_-]+)", url)
                if match:
                    filename = match.group(1)
                else:
                    return None

            with open(f'./{filename}.html', 'r', encoding='utf-8') as html:
                bs = BeautifulSoup(html.read(), 'html.parser')
            return bs
        
        except URLError as e:
            print(f"URL error: {e.reason}")
            return None 

In [169]:
bs = webscraper.get_bs_from_url()

HTTP error: 429 - Too Many Requests
Loading from file


## 1. Title of the publications since 2015

In [170]:
publication_titles_since_2015 = list()
publication_links_since_2015 = list()

tr = 'gsc_a_tr',
td = 'gsc_a_y',
a = 'gsc_a_at',

for class_name in bs.find_all('tr', class_ = tr):

    # Filter publication year
    year = int(class_name.find('td', class_ = td).span.get_text())
    if year < 2015:
        continue

    # Get publications
    publication_titles_since_2015.append(class_name.find('a').get_text())

    # Get links
    a_tag = class_name.find('a', class_ = a, href = True)
    if a_tag:
        publication_links_since_2015.append(urljoin(base_url, a_tag['href']))

print("All publications since 2015:")
print('\n'.join(publication_titles_since_2015))

All publications since 2015:
Designing for exploratory play with a hackable digital musical instrument
Making music together: an exploration of amateur and pro-am Grime music production
Understanding social media and sound: music, meaning and membership, the case of SoundCloud
GeoTracks: Adaptive music for everyday journeys
The user experience of mobile music making: An ethnographic exploration of music production and performance in practice
The Grime scene: social media, music, creation and consumption
The Rough Mile: Reframing Location Through Locative Audio
The Rough Mile: Testing a framework of immersive practice
An ethnographic exploration of studio production practice
Breaking the workflow: Design heuristics to support the development of usable digital audio production tools: framing usability heuristics for contemporary purposes
DESIGNING AND DEVELOPING USER-CENTRED SYSTEMS
Evaluating colour in concept diagrams
The Rough Mile: A Design Template for Locative Audio Experiences
Eva

In [171]:
publication_links_since_2015

['https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:u-x6o8ySG0sC',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:UeHWp8X0CEIC',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:u5HHmVD_uO8C',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:d1gkVwhDpl0C',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:IjCSPb-OGe4C',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:qjMakFHDy7sC',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAAJ&citation_for_view=ETIBghkAAAAJ:W7OEmFMy1HYC',
 'https://scholar.google.com/citations?view_op=view_citation&hl=en&user=ETIBghkAAAA

## 2. Co-authors and their number

In [172]:
coauthors_map = {}

for i, link in enumerate(publication_links_since_2015):
    
    # Limit if there was a http error
    if i == url_limit:
        break
    
    bs = webscraper.get_bs_from_url(link)
    
    # Find authors section
    authors = bs.find('div', string=re.compile('Authors', re.IGNORECASE))
    if authors:
        authors_div = authors.find_next_sibling('div')
        coauthors_map[publication_titles_since_2015[i]] = [author.strip() for author in authors_div.get_text().split(',')
                                                                if author.strip() != "Sean McGrath"]

HTTP error: 429 - Too Many Requests
Loading from file
HTTP error: 429 - Too Many Requests
Loading from file
HTTP error: 429 - Too Many Requests
Loading from file


In [173]:
for key in coauthors_map:
    print(f"Title: {key}")
    print("Co-Authors: " + ", ".join(coauthors_map[key]))
    coauthor_num = len(coauthors_map[key])
    print(f"Number: {coauthor_num}")

Title: Designing for exploratory play with a hackable digital musical instrument
Co-Authors: Andrew P McPherson, Alan Chamberlain, Adrian Hazzard, Steve Benford
Number: 4
Title: Making music together: an exploration of amateur and pro-am Grime music production
Co-Authors: Alan Chamberlain, Steve Benford
Number: 2
Title: Understanding social media and sound: music, meaning and membership, the case of SoundCloud
Co-Authors: Alan Chamberlain, Steve Benford
Number: 2
